# Eco-Logistics GRPO Training — v2 (hackathon submission)

**Theme:** #3.1 World Modeling / Professional Tasks (primary), #2 Long-Horizon Planning (secondary).

**What this notebook does:**
1. Installs TRL + OpenEnv + Unsloth
2. Connects to the hosted `eco-logistics` HF Space (configured for parallel rollouts)
3. Wraps the env with **non-stationary demand shocks** and a **competitor agent** that bids up shipping prices — strengthens the world-modeling story without pretending to be multi-agent
4. Normalizes the full dense reward (profit − shipping − carbon − storage + healthy-stock bonus) for GRPO
5. Logs 4 metrics per episode: total reward, profit, carbon usage, delivery success rate
6. Runs GRPO training
7. Produces 2 training curves (reward vs episodes, carbon efficiency vs episodes) for the pitch deck
8. Compares before/after agent behavior

**Hardware:** single T4 (free Colab) or A100 (HF credits). Uses Unsloth + LoRA to fit in <15GB VRAM.

## 1. Setup

In [ ]:
!pip install -Uq "trl[vllm]" openenv-core
!pip install -Uq unsloth trackio
!pip install -Uq matplotlib pandas numpy

In [ ]:
from huggingface_hub import login
import os

HF_TOKEN = os.environ.get("HF_TOKEN")
if HF_TOKEN:
    login(token=HF_TOKEN)
else:
    print("WARNING: set HF_TOKEN before pushing the final model.")

## 2. Configuration

In [ ]:
# ---- Point this at your deployed HF Space ----
# IMPORTANT: your main.py on the Space must set max_concurrent_envs >= 8
# (see TRAINING_PLAN.md for the one-line diff)
ENV_URL = os.environ.get("ENV_URL", "https://YOUR_USERNAME-eco-logistics.hf.space")

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

# Start on easy task to confirm the pipeline learns; then re-run on harder tasks.
TASK_ID = "restock_only"

# GRPO hyperparameters
NUM_GENERATIONS = 4
PER_DEVICE_BATCH = 1
GRAD_ACCUM_STEPS = 4
MAX_STEPS = 100
MAX_COMPLETION_LENGTH = 512
MAX_ENV_STEPS = 10
LEARNING_RATE = 5e-6

# World-modeling wrapper knobs (these run client-side, no env changes needed)
DEMAND_SHOCK_PROB = 0.15   # prob of a demand spike at any step
DEMAND_SHOCK_MULT = 2.5    # 2.5x normal demand when shocked
COMPETITOR_BID_PROB = 0.20 # prob competitor bids up a random route this step
COMPETITOR_BID_MULT = 1.8  # 1.8x shipping cost on the contested route

# Required concurrency for parallel rollouts
REQUIRED_CONCURRENT = NUM_GENERATIONS * PER_DEVICE_BATCH * GRAD_ACCUM_STEPS
print(f"HF Space must support max_concurrent_envs >= {REQUIRED_CONCURRENT}")
print(f"Training {MODEL_NAME} on task={TASK_ID} against {ENV_URL}")

## 3. Smoke-test the env

In [ ]:
import requests

r = requests.post(f"{ENV_URL}/reset", json={"task_id": TASK_ID}, timeout=30)
r.raise_for_status()
initial_obs = r.json()
print("Reset OK. Observation keys:", list(initial_obs.keys()))

dummy_action = {
    "ship_amount": 0.0,
    "origin_city": "Seattle",
    "destination_city": "Chicago",
    "speed_mode": "Rail",
}
r = requests.post(f"{ENV_URL}/step", json=dummy_action, timeout=30)
r.raise_for_status()
step_resp = r.json()
print("Step OK. Reward breakdown:", step_resp.get("reward"))

## 4. World-modeling wrapper: non-stationary demand + competitor

These two knobs add genuine world-modeling complexity on top of the base env. They run **client-side** so we don't need to redeploy the Space:

- **Demand shock**: with probability `DEMAND_SHOCK_PROB` per step, multiply the observed `current_demand` by `DEMAND_SHOCK_MULT`. The agent sees the spike in its prompt and must react (ship more, use Air mode, etc).
- **Competitor bid**: with probability `COMPETITOR_BID_PROB`, a random route becomes 1.8x more expensive for that step. We surface this as a `weather_alert`-style warning so the agent has to route around it.

These change the **observation the agent sees** and the **reward it receives** after the env responds — they do NOT require modifying the environment itself. This keeps us Round-1-compliant while honestly increasing task difficulty.

In [ ]:
import random

CITIES = ["Seattle", "Chicago", "NYC"]

def apply_world_wrapper(obs: dict, rng: random.Random) -> tuple[dict, dict]:
    """Apply demand shock + competitor bid to an observation before showing it to the agent.
    
    Returns (modified_obs, shock_state) where shock_state holds the per-step modifiers
    we'll use to adjust the reward post-hoc.
    """
    modified = dict(obs)
    shock_state = {"demand_shocked": False, "contested_route": None}
    
    # Demand shock
    if rng.random() < DEMAND_SHOCK_PROB:
        shocked_demand = {c: v * DEMAND_SHOCK_MULT for c, v in obs["current_demand"].items()}
        modified["current_demand"] = shocked_demand
        shock_state["demand_shocked"] = True
    
    # Competitor bid — surface as a weather_alert-style note
    if rng.random() < COMPETITOR_BID_PROB:
        origin = rng.choice(CITIES)
        dest = rng.choice([c for c in CITIES if c != origin])
        shock_state["contested_route"] = (origin, dest)
        existing = modified.get("weather_alert") or ""
        note = f"Competitor bid up {origin}→{dest} route ({COMPETITOR_BID_MULT}x cost this step)"
        modified["weather_alert"] = f"{existing}; {note}" if existing else note
    
    return modified, shock_state


def apply_reward_wrapper(reward_dict: dict, action: dict, shock_state: dict) -> dict:
    """Adjust reward components based on the shocks that were active this step.
    Returns a modified reward dict with a new 'total' field.
    """
    adjusted = dict(reward_dict)
    
    # If the agent shipped on a contested route, penalize extra shipping cost
    contested = shock_state.get("contested_route")
    if contested and action.get("ship_amount", 0) > 0:
        if (action.get("origin_city"), action.get("destination_city")) == contested:
            extra_cost = adjusted.get("shipping_cost", 0) * (COMPETITOR_BID_MULT - 1.0)
            adjusted["shipping_cost"] = adjusted.get("shipping_cost", 0) + extra_cost
    
    # Recompute total with the same formula the env uses
    adjusted["total"] = (
        adjusted.get("sales_revenue", 0)
        - adjusted.get("shipping_cost", 0)
        - adjusted.get("carbon_penalty", 0)
        - adjusted.get("storage_fee", 0)
        + adjusted.get("healthy_stock_bonus", 0)
    )
    return adjusted

## 5. System prompt + action parser

In [ ]:
SYSTEM_PROMPT = """You are an AI supply chain manager for a 3-warehouse network (Seattle, Chicago, NYC).

Each step you choose ONE shipping action. Your goal: maximize profit while keeping carbon emissions low.

Reply with exactly one line of valid JSON matching this schema:
{"ship_amount": <float>=0>, "origin_city": <"Seattle"|"Chicago"|"NYC">, "destination_city": <"Seattle"|"Chicago"|"NYC">, "speed_mode": <"Air"|"Rail">}

Rules:
- Rail: cheap, low-carbon, slow (3 steps).
- Air: fast (1 step), expensive, high-carbon.
- ship_amount=0 is a valid no-op.
- The weather_alert field may warn about disrupted or contested routes (higher cost) — route around them.
- Demand can spike unexpectedly — watch for sudden jumps in current_demand.

Respond with ONLY the JSON object. No explanation, no markdown."""

def format_observation_prompt(obs: dict) -> str:
    return (
        f"Step {obs['step_number']}/{obs['total_steps']}\n"
        f"Inventory: {obs['current_inventory']}\n"
        f"Demand this step: {obs['current_demand']}\n"
        f"Pending shipments: {obs.get('pending_shipments', [])}\n"
        f"Carbon credits left: {obs['carbon_credit_balance']:.1f}\n"
        f"Cumulative profit: {obs.get('cumulative_profit', 0):.2f}\n"
        f"Weather/market alert: {obs.get('weather_alert') or 'clear'}\n"
        f"\nYour action (JSON only):"
    )

In [ ]:
import json
import re

SAFE_FALLBACK_ACTION = {
    "ship_amount": 0.0,
    "origin_city": "Seattle",
    "destination_city": "Chicago",
    "speed_mode": "Rail",
}

def parse_action(completion: str) -> tuple[dict, bool]:
    m = re.search(r"\{[^{}]*\}", completion, re.DOTALL)
    if not m:
        return SAFE_FALLBACK_ACTION, False
    try:
        action = json.loads(m.group(0))
        required = {"ship_amount", "origin_city", "destination_city", "speed_mode"}
        if not required.issubset(action.keys()):
            return SAFE_FALLBACK_ACTION, False
        return action, True
    except (json.JSONDecodeError, ValueError):
        return SAFE_FALLBACK_ACTION, False

## 6. Episode rollout with full metric logging

Returns the 4 metrics the judges want to see:
- `total_reward` — summed normalized reward (what GRPO optimizes)
- `profit` — summed sales_revenue − shipping_cost (business metric)
- `carbon_used` — cumulative carbon emissions (sustainability metric)
- `delivery_success_rate` — fraction of demand fulfilled (operations metric)
- `valid_action_rate` — fraction of model outputs that parsed as valid JSON (diagnostic)

In [ ]:
import uuid

def run_episode(
    generate_fn,
    task_id: str = TASK_ID,
    max_steps: int = MAX_ENV_STEPS,
    env_url: str = ENV_URL,
    seed: int | None = None,
):
    """Run one full episode and return detailed metrics.
    
    Uses a unique X-Session-Id header so parallel rollouts don't share env state.
    Your main.py must be the v2 version that supports the session pool.
    """
    rng = random.Random(seed)
    session_id = f"train-{uuid.uuid4().hex[:12]}"
    headers = {"X-Session-Id": session_id}
    
    r = requests.post(f"{env_url}/reset", json={"task_id": task_id}, headers=headers, timeout=30)
    r.raise_for_status()
    obs = r.json()
    
    trajectory = []
    total_reward = 0.0
    total_profit = 0.0           # sales_revenue - shipping_cost
    total_carbon = 0.0           # cumulative carbon_penalty (proxy for emissions)
    total_demand = 0.0
    total_fulfilled = 0.0
    valid_count = 0
    
    for step_idx in range(max_steps):
        # Apply world-modeling wrapper to observation
        wrapped_obs, shock_state = apply_world_wrapper(obs, rng)
        
        # Generate action
        user_prompt = format_observation_prompt(wrapped_obs)
        full_prompt = f"<|system|>\n{SYSTEM_PROMPT}\n<|user|>\n{user_prompt}\n<|assistant|>\n"
        completion = generate_fn(full_prompt)
        action, was_valid = parse_action(completion)
        if was_valid:
            valid_count += 1
        
        # Step env
        r = requests.post(f"{env_url}/step", json=action, headers=headers, timeout=30)
        r.raise_for_status()
        step_resp = r.json()
        
        # Apply reward wrapper for competitor bid penalty
        raw_reward = step_resp.get("reward", {})
        adjusted_reward = apply_reward_wrapper(raw_reward, action, shock_state)
        step_reward = adjusted_reward.get("total", 0.0)
        
        # Accumulate detailed metrics
        total_reward += step_reward
        total_profit += (adjusted_reward.get("sales_revenue", 0)
                         - adjusted_reward.get("shipping_cost", 0))
        total_carbon += adjusted_reward.get("carbon_penalty", 0)
        
        # Delivery success: compare fulfilled demand (inferred from sales_revenue)
        # vs total demand this step
        step_demand = sum(wrapped_obs["current_demand"].values())
        total_demand += step_demand
        # Assume revenue scales 1:1 with fulfilled units for simplicity;
        # adjust divisor if your env uses a different unit price
        UNIT_PRICE = 10.0  # match env.py's price per unit
        fulfilled_units = adjusted_reward.get("sales_revenue", 0) / UNIT_PRICE
        total_fulfilled += fulfilled_units
        
        obs = step_resp.get("observation", obs)
        trajectory.append({
            "step": step_idx,
            "action": action,
            "valid": was_valid,
            "reward": step_reward,
            "profit": adjusted_reward.get("sales_revenue", 0) - adjusted_reward.get("shipping_cost", 0),
            "carbon": adjusted_reward.get("carbon_penalty", 0),
            "shocked": shock_state["demand_shocked"],
            "contested": shock_state["contested_route"] is not None,
        })
        
        if step_resp.get("done"):
            break
    
    # Grader score for the task
    try:
        g = requests.post(f"{env_url}/grader", json={"task_id": task_id}, headers=headers, timeout=30)
        grader_score = g.json().get("score", 0.0)
    except Exception:
        grader_score = 0.0
    
    return {
        "total_reward": total_reward,
        "profit": total_profit,
        "carbon_used": total_carbon,
        "carbon_efficiency": total_profit / max(1.0, total_carbon),  # $ profit per unit carbon
        "delivery_success_rate": total_fulfilled / max(1.0, total_demand),
        "valid_action_rate": valid_count / max(1, len(trajectory)),
        "grader_score": grader_score,
        "trajectory": trajectory,
    }

## 7. Load model + baseline eval

In [ ]:
from unsloth import FastLanguageModel
import torch
import numpy as np

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=2048,
    load_in_4bit=True,
    dtype=None,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0.0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

In [ ]:
def eager_generate(prompt: str) -> str:
    FastLanguageModel.for_inference(model)
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=128,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            pad_token_id=tokenizer.eos_token_id,
        )
    completion = tokenizer.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    return completion

print("Running 10 BASELINE episodes...")
baseline_results = [run_episode(eager_generate, seed=i) for i in range(10)]

def summarize(results, label):
    print(f"\n=== {label} ===")
    for key in ["total_reward", "profit", "carbon_used", "carbon_efficiency",
                "delivery_success_rate", "valid_action_rate", "grader_score"]:
        vals = [r[key] for r in results]
        print(f"  {key:25s}: {np.mean(vals):+8.3f} ± {np.std(vals):.3f}")

summarize(baseline_results, "BEFORE TRAINING")

## 8. GRPO training with normalized dense reward

Key points:
- **Reward normalization**: we divide episode returns by a running std to give GRPO stable advantages. GRPO's group-relative advantage already normalizes within a group; this extra normalization helps across groups.
- **Full dense signal preserved**: `total_reward` is computed from all four components (revenue − shipping − carbon − storage + bonus) plus our wrapper adjustments.
- **Per-step metric logging**: we record profit/carbon/delivery alongside reward so the training curves show tradeoffs, not just one number.

In [ ]:
from trl import GRPOConfig, GRPOTrainer
from datasets import Dataset

train_dataset = Dataset.from_list([
    {"prompt": f"task:{TASK_ID}"} for _ in range(MAX_STEPS * GRAD_ACCUM_STEPS)
])

In [ ]:
# Running stats for reward normalization across GRPO steps
class RunningStats:
    def __init__(self, eps=1e-6):
        self.n = 0
        self.mean = 0.0
        self.m2 = 0.0
        self.eps = eps
    def update(self, x):
        self.n += 1
        delta = x - self.mean
        self.mean += delta / self.n
        self.m2 += delta * (x - self.mean)
    @property
    def std(self):
        return max(self.eps, (self.m2 / max(1, self.n - 1)) ** 0.5)

reward_stats = RunningStats()

# Full training log — dumped to CSV at the end for the pitch
training_log = {
    "grpo_step":      [],
    "mean_reward":    [],
    "mean_profit":    [],
    "mean_carbon":    [],
    "carbon_efficiency": [],
    "delivery_rate":  [],
    "valid_rate":     [],
    "grader_score":   [],
}
global_step_counter = {"n": 0}

In [ ]:
def rollout_func(prompts, trainer):
    """Custom rollout: for each prompt, run a full episode against the env,
    log all 4 business metrics, attach normalized reward for GRPO."""
    all_prompt_ids, all_completion_ids, all_logprobs, all_rewards = [], [], [], []
    batch_metrics = {"reward": [], "profit": [], "carbon": [],
                     "carbon_eff": [], "delivery": [], "valid": [], "grader": []}
    
    for _ in prompts:
        result = run_episode(eager_generate)
        
        # Normalize the raw episode return
        raw_reward = result["total_reward"]
        reward_stats.update(raw_reward)
        normalized_reward = (raw_reward - reward_stats.mean) / reward_stats.std
        
        full_completion = "\n".join(
            json.dumps(t["action"]) for t in result["trajectory"]
        )
        ids = tokenizer(full_completion, return_tensors="pt").input_ids[0]
        all_completion_ids.append(ids.tolist())
        all_prompt_ids.append(tokenizer(SYSTEM_PROMPT, return_tensors="pt").input_ids[0].tolist())
        all_logprobs.append([0.0] * len(ids))
        all_rewards.append(normalized_reward)
        
        batch_metrics["reward"].append(raw_reward)
        batch_metrics["profit"].append(result["profit"])
        batch_metrics["carbon"].append(result["carbon_used"])
        batch_metrics["carbon_eff"].append(result["carbon_efficiency"])
        batch_metrics["delivery"].append(result["delivery_success_rate"])
        batch_metrics["valid"].append(result["valid_action_rate"])
        batch_metrics["grader"].append(result["grader_score"])
    
    # Log batch means
    global_step_counter["n"] += 1
    training_log["grpo_step"].append(global_step_counter["n"])
    training_log["mean_reward"].append(np.mean(batch_metrics["reward"]))
    training_log["mean_profit"].append(np.mean(batch_metrics["profit"]))
    training_log["mean_carbon"].append(np.mean(batch_metrics["carbon"]))
    training_log["carbon_efficiency"].append(np.mean(batch_metrics["carbon_eff"]))
    training_log["delivery_rate"].append(np.mean(batch_metrics["delivery"]))
    training_log["valid_rate"].append(np.mean(batch_metrics["valid"]))
    training_log["grader_score"].append(np.mean(batch_metrics["grader"]))
    
    return {
        "prompt_ids": all_prompt_ids,
        "completion_ids": all_completion_ids,
        "logprobs": all_logprobs,
        "env_reward": all_rewards,
    }

def env_reward_func(completions, env_reward, **kwargs):
    return list(env_reward)

In [ ]:
training_args = GRPOConfig(
    output_dir="./eco-logistics-grpo-v2",
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=PER_DEVICE_BATCH,
    gradient_accumulation_steps=GRAD_ACCUM_STEPS,
    num_generations=NUM_GENERATIONS,
    max_completion_length=MAX_COMPLETION_LENGTH,
    max_prompt_length=1024,
    max_steps=MAX_STEPS,
    logging_steps=1,
    save_steps=50,
    bf16=True,
    use_vllm=False,  # flip to True on A100 after pipeline confirmed
    report_to="trackio",
    run_name=f"eco-logistics-v2-{TASK_ID}",
)

trainer = GRPOTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    reward_funcs=env_reward_func,
    rollout_func=rollout_func,
    processing_class=tokenizer,
)

trainer.train()

## 9. Post-training evaluation

In [ ]:
print("Running 10 AFTER episodes...")
trained_results = [run_episode(eager_generate, seed=i+1000) for i in range(10)]
summarize(baseline_results, "BEFORE TRAINING")
summarize(trained_results, "AFTER TRAINING")

print("\n=== DELTAS ===")
for key in ["total_reward", "profit", "carbon_used",
            "carbon_efficiency", "delivery_success_rate", "grader_score"]:
    before = np.mean([r[key] for r in baseline_results])
    after = np.mean([r[key] for r in trained_results])
    print(f"  {key:25s}: {before:+8.3f} → {after:+8.3f}  (Δ {after-before:+.3f})")

## 10. Training curves for the pitch deck

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

log_df = pd.DataFrame(training_log)
log_df.to_csv(f"training_log_{TASK_ID}.csv", index=False)
print(f"Saved training_log_{TASK_ID}.csv")

# Simple rolling mean for smoother curves
WINDOW = 5

fig, axes = plt.subplots(2, 2, figsize=(13, 9))

# (1) Reward vs episodes
axes[0, 0].plot(log_df["grpo_step"], log_df["mean_reward"], alpha=0.3, label="per step")
axes[0, 0].plot(log_df["grpo_step"], log_df["mean_reward"].rolling(WINDOW).mean(),
                linewidth=2, label=f"{WINDOW}-step rolling mean")
axes[0, 0].set_xlabel("GRPO step")
axes[0, 0].set_ylabel("Mean episode reward")
axes[0, 0].set_title(f"Reward vs episodes — {TASK_ID}")
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# (2) Carbon efficiency vs episodes (the tradeoff story)
axes[0, 1].plot(log_df["grpo_step"], log_df["carbon_efficiency"], alpha=0.3, label="per step")
axes[0, 1].plot(log_df["grpo_step"], log_df["carbon_efficiency"].rolling(WINDOW).mean(),
                linewidth=2, color="#2a9d8f", label=f"{WINDOW}-step rolling mean")
axes[0, 1].set_xlabel("GRPO step")
axes[0, 1].set_ylabel("Profit per unit carbon")
axes[0, 1].set_title("Carbon efficiency vs episodes")
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# (3) Before/after grader score
before_g = [r["grader_score"] for r in baseline_results]
after_g = [r["grader_score"] for r in trained_results]
axes[1, 0].bar(["Before", "After"], [np.mean(before_g), np.mean(after_g)],
               yerr=[np.std(before_g), np.std(after_g)],
               capsize=6, color=["#888", "#2a9d8f"])
axes[1, 0].set_ylabel("Grader score (0-1)")
axes[1, 0].set_title("Task completion: before vs after")
axes[1, 0].set_ylim(0, 1)

# (4) Delivery success + profit side-by-side (before/after)
x = np.arange(2)
width = 0.35
delivery_before = np.mean([r["delivery_success_rate"] for r in baseline_results])
delivery_after = np.mean([r["delivery_success_rate"] for r in trained_results])
# Normalize profit to [0,1] range by dividing by max for visualization on same axis
profit_before = np.mean([r["profit"] for r in baseline_results])
profit_after = np.mean([r["profit"] for r in trained_results])
profit_norm = max(abs(profit_before), abs(profit_after), 1)
axes[1, 1].bar(x - width/2, [delivery_before, delivery_after], width, label="Delivery rate")
axes[1, 1].bar(x + width/2, [profit_before/profit_norm, profit_after/profit_norm],
               width, label=f"Profit (norm. by {profit_norm:.0f})")
axes[1, 1].set_xticks(x)
axes[1, 1].set_xticklabels(["Before", "After"])
axes[1, 1].set_title("Operations metrics: before vs after")
axes[1, 1].legend()

plt.tight_layout()
plt.savefig(f"training_curves_{TASK_ID}.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved training_curves_{TASK_ID}.png for the pitch deck.")

## 11. Save qualitative trajectories (before / after example runs)

Two text files the judges can read — one baseline episode and one trained episode — showing the agent's actual decision-making.

In [ ]:
def dump_trajectory(result, path):
    with open(path, "w") as f:
        f.write(f"Task: {TASK_ID}\n")
        f.write(f"Total reward: {result['total_reward']:.2f}\n")
        f.write(f"Profit: {result['profit']:.2f}\n")
        f.write(f"Carbon used: {result['carbon_used']:.2f}\n")
        f.write(f"Delivery rate: {result['delivery_success_rate']:.1%}\n")
        f.write(f"Grader score: {result['grader_score']:.3f}\n\n")
        for t in result["trajectory"]:
            flags = []
            if t["shocked"]: flags.append("DEMAND_SHOCK")
            if t["contested"]: flags.append("COMPETITOR_BID")
            flag_str = f" [{', '.join(flags)}]" if flags else ""
            f.write(f"Step {t['step']}{flag_str}: {t['action']}\n")
            f.write(f"  reward={t['reward']:+.2f}  profit={t['profit']:+.2f}  carbon={t['carbon']:.2f}  valid={t['valid']}\n")
    print(f"Wrote {path}")

# Use the median-performing episode from each set (more representative than best/worst)
def median_idx(results, key="total_reward"):
    vals = [r[key] for r in results]
    sorted_idx = sorted(range(len(vals)), key=lambda i: vals[i])
    return sorted_idx[len(sorted_idx) // 2]

dump_trajectory(baseline_results[median_idx(baseline_results)], f"trajectory_before_{TASK_ID}.txt")
dump_trajectory(trained_results[median_idx(trained_results)], f"trajectory_after_{TASK_ID}.txt")

## 12. Save + push the LoRA adapter

In [ ]:
model.save_pretrained(f"eco-logistics-lora-v2-{TASK_ID}")
tokenizer.save_pretrained(f"eco-logistics-lora-v2-{TASK_ID}")

# Optional: push to HF Hub
# model.push_to_hub(f"YOUR_USERNAME/eco-logistics-lora-v2-{TASK_ID}")
# tokenizer.push_to_hub(f"YOUR_USERNAME/eco-logistics-lora-v2-{TASK_ID}")

## What's different from v1

| Change | Why |
|---|---|
| Full dense reward kept + normalized via running mean/std | Preserves all 4 reward components; running-std normalization gives GRPO stable advantages across training |
| World-modeling wrapper (demand shocks + competitor bids) | Honest complexity upgrade. Strengthens Theme #3.1 narrative without pretending to be multi-agent |
| 4 logged metrics per episode (reward, profit, carbon, delivery) | Judges' 20% "improvement" score rewards observable metrics — one number isn't enough |
| 2 training curves (reward + carbon efficiency) | Tells the tradeoff story: agent learns to earn more *per unit carbon*, not just more profit |
| Before/after median-episode trajectory dumps | Qualitative evidence of behavior change — text files the judges can skim |
| Concurrency guard (prints required `max_concurrent_envs`) | Catches the #1 reason GRPO silently stalls on OpenEnv |

## Next after this runs

1. Confirm the reward curve trends up on `restock_only`
2. Re-run on `inventory_balanced` and `net_zero_profit` — the hard task is where the pitch lives
3. Compile the 4 PNGs + 6 trajectory TXTs + 3 CSVs into a submission folder
4. Record the <2 min video: show `training_curves_net_zero_profit.png` and narrate the delta